In [ ]:
# CyberSentinel ML Baseline
# Load data, preprocess, train a simple model, and report accuracy

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

# 1) Load CSV
csv_path = 'data/phishing.csv'
df = pd.read_csv(csv_path)
print('Loaded rows:', len(df))
df.head()


In [ ]:
# 2) Cleanup: drop all-null columns and fill NaNs

df = df.dropna(axis=1, how='all')
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].fillna('')
    else:
        df[col] = df[col].fillna(0)

assert 'label' in df.columns, f"Expected 'label' in columns, found: {df.columns.tolist()}"

text_cols = ['title', 'description']
cat_cols = ['source']
y = df['label']
X = df[text_cols + cat_cols]


In [4]:
# 3) Preprocess and split

def combine_text(X_df: pd.DataFrame):
    return (X_df[text_cols[0]].astype(str) + ' ' + X_df[text_cols[1]].astype(str)).values

text_combiner = FunctionTransformer(combine_text, validate=False)
text_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
text_pipeline = Pipeline([
    ('combine', text_combiner),
    ('tfidf', text_vectorizer),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_pipeline, text_cols + cat_cols),  # combine reads text_cols
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ],
    remainder='drop'
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if len(y.unique()) > 1 else None
)
print('Train size:', len(X_train), 'Test size:', len(X_test))


NameError: name 'pd' is not defined

In [ ]:
# 4) Train and evaluate

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', model),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print('Accuracy:', round(acc, 4))
print(classification_report(y_test, y_pred))


In [ ]:
# 5) Optional: save model

import joblib
joblib.dump(pipeline, 'model.pkl')
print('Saved to model.pkl')
